# 07 Domain presets end to end

AFX ships domain presets (plus `egemaps`), validated configs that `afx.load_preset(name)` returns and every entry point accepts. Set `DOMAIN` and run all: the notebook extracts the preset's features, trains a scikit-learn classifier and plots a confusion matrix, feature importances and a PCA view.

| `DOMAIN` | Preset | Dataset | Task |
|---|---|---|---|
| `speech` | Speech & paralinguistics | RAVDESS (speech audio) | emotion classification |
| `bioacoustics` | Bioacoustics | ESC-50 (animal subset) | animal sound classification |
| `music` | Music & MIR | GTZAN | genre classification |
| `machinery` | Machinery & condition monitoring | DCASE 2020 Task 2 dev (pump) | normal vs anomaly |

With `DATASETS=/some/dir` set, the domain's dataset downloads there once (210 MB to 1.2 GB). Without it, with `AFX_SYNTHETIC=1`, or if the download fails, synthetic stand-in clips from `demo_data.py` are used so the notebook still runs; the numbers then say nothing about the domain. Dataset logic lives in `domain_packs.py`.

In [ ]:
import tempfile
import time
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import afx_audio as afx
import domain_packs as dp

DOMAIN = 'speech'      # 'speech' | 'bioacoustics' | 'music' | 'machinery'
N_PER_CLASS = 60       # clips per class (None = all)
CLASSES = None         # None = the domain's default subset
MAX_SECONDS = 8.0      # read at most this much of each clip
TEST_SIZE = 0.3
SEED = 0

spec = dp.DOMAINS[DOMAIN]
classes = list(CLASSES or spec.default_classes or []) or None
print(f'{spec.title}: {spec.task}')
print(f'dataset {spec.dataset_name} (~{spec.approx_download_mb} MB), classes {classes}')

## The preset

A preset is a config dict: `sample_rate`, a `features` block with domain-tuned frame and hop sizes, and an `aggregation`.

In [ ]:
config = afx.load_preset(spec.preset)
print(afx.preset_info(spec.preset)['description'], '\n')
print(f"sample_rate {config['sample_rate']} Hz, aggregation {config['aggregation']!r}")
for name, params in config['features'].items():
    print(f'  {name:20s} {params}')

## Data

In [ ]:
tmp = tempfile.mkdtemp(prefix='afx_domain_')
t0 = time.time()
examples, source = dp.collect_examples(spec, classes, N_PER_CLASS, SEED, synthetic_dir=tmp,
                                       sr=config['sample_rate'])
print(f'{len(examples)} clips from {source} in {time.time() - t0:.1f} s')
print(dict(sorted(Counter(label for _, label in examples).items())))

## Features

`extract_all_features` applies the preset's parameters and aggregation, so every clip becomes one vector with the same named columns.

In [ ]:
t0 = time.time()
X, y, feature_names = dp.build_feature_matrix(examples, config, max_seconds=MAX_SECONDS)
print(f'X {X.shape} in {time.time() - t0:.1f} s; NaN cells: {int(np.isnan(X).sum())}')
print('first columns:', feature_names[:6])

## Classifier

Median imputation for clip-level NaN, standardization, and an untuned random forest.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
clf = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
)
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)
print(f'accuracy {accuracy_score(y_te, y_pred):.3f} (chance {1 / len(set(y)):.3f})\n')
print(classification_report(y_te, y_pred, zero_division=0))

In [ ]:
labels = sorted(set(y))
fig, ax = plt.subplots(figsize=(1.2 * len(labels) + 2, 1.2 * len(labels) + 1))
ConfusionMatrixDisplay.from_predictions(y_te, y_pred, labels=labels, cmap='Blues', colorbar=False,
                                        xticks_rotation=45, ax=ax)
ax.set_title(f'{spec.title}: confusion matrix')
fig.tight_layout()

## Which columns carry the signal

In [ ]:
forest = clf.named_steps['randomforestclassifier']
order = np.argsort(forest.feature_importances_)[::-1][:15]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(range(order.size), forest.feature_importances_[order][::-1])
ax.set_yticks(range(order.size), [feature_names[i] for i in order][::-1], fontsize=9)
ax.set_xlabel('impurity importance')
ax.set_title(f'{spec.title}: top 15 columns')
fig.tight_layout()

## PCA view

In [ ]:
Xs = make_pipeline(SimpleImputer(strategy='median'), StandardScaler()).fit_transform(X)
pcs = PCA(n_components=2, random_state=SEED).fit_transform(Xs)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for label in labels:
    m = y == label
    ax.scatter(pcs[m, 0], pcs[m, 1], s=18, alpha=0.7, label=label)
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.set_title(f'{spec.title}: PCA of preset features')
ax.legend(fontsize=8)
fig.tight_layout()

## Other domains and custom presets

Change `DOMAIN` and run all again. A preset is a JSON file in `afx_audio/presets/`; a dict with the same keys works anywhere a preset name does (`extract_all_features`, `extract_features_from_folder`, `AFXFeatureExtractor(config=...)`, `afx --config`).

In [ ]:
import shutil

shutil.rmtree(tmp)